# 🐾 KXD — Generador Furry & YiFF

Antes de empezar:
1. Activá GPU gratis → menú **Runtime › Change runtime type › T4 GPU**
2. Ejecutá la **Celda 1** de abajo *(solo la primera vez — instala todo)*
3. Cuando termine, ejecutá la **Celda 2** para conectarte a la web

---

## ▶ Celda 1 — Instalación y descarga de modelos

Ejecutá esta celda **una sola vez**. Instala las dependencias y descarga el modelo SDXL y todos los LoRAs.  
Tarda unos **5 minutos** la primera vez. Las siguientes veces es instantáneo porque ya están guardados.

Cuando veas el mensaje ✅ **Todo listo**, pasá a la Celda 2.

In [ ]:
import subprocess, sys, os, requests

# 🔑 PEGA AQUÍ TU API KEY DE CIVITAI
CIVITAI_API_KEY = "3a0a1d671ce384e4a599a84c51b8d250"

print('📦 Instalando dependencias...')
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q', '--upgrade', '--force-reinstall',
    'torchao'
], check=True)

subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q',
    'diffusers>=0.27.0', 'transformers', 'accelerate', 'safetensors',
    'torch', 'torchvision', 'xformers',
    'fastapi', 'uvicorn[standard]', 'requests', 'Pillow',
], check=True)

if not os.path.exists('/usr/local/bin/cloudflared'):
    subprocess.run(['wget', '-q', '-O', '/tmp/cloudflared.deb',
        'https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb'], check=True)
    subprocess.run(['dpkg', '-i', '/tmp/cloudflared.deb'], check=True)
print('✅ Dependencias listas')

os.makedirs('/content/models', exist_ok=True)
os.makedirs('/content/loras',  exist_ok=True)

MODEL_PATH = '/content/models/sdxl_model.safetensors'
LORA_REGISTRY = {
    'zonkpunch':   '/content/loras/zonkpunch.safetensors',
    'zcik':        '/content/loras/zcik.safetensors',
    'danziEngine': '/content/loras/danzi_engine.safetensors',
    'edjit':       '/content/loras/edjit.safetensors',
    'dagasi':      '/content/loras/dagasi.safetensors',
}

DOWNLOADS = [
    ('https://civitai.com/api/download/models/2074658?fileId=1970963', MODEL_PATH,                   'Modelo SDXL'),
    ('https://civitai.red/api/download/models/2949119?fileId=2828573', LORA_REGISTRY['zonkpunch'],   'LoRA Zonkpunch'),
    ('https://civitai.com/api/download/models/1340337?fileId=1243946', LORA_REGISTRY['zcik'],        'LoRA Zcik'),
    ('https://civitai.com/api/download/models/3137729?fileId=3017792', LORA_REGISTRY['danziEngine'], 'LoRA Danzi Engine'),
    ('https://civitai.com/api/download/models/2729690?fileId=2615709', LORA_REGISTRY['edjit'],       'LoRA Edjit'),
    ('https://civitai.com/api/download/models/2307358?fileId=2198032', LORA_REGISTRY['dagasi'],      'LoRA Dagasi'),
]

headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64)',
}
if CIVITAI_API_KEY:
    headers['Authorization'] = f'Bearer {CIVITAI_API_KEY}'
else:
    print('⚠️ ADVERTENCIA: No has puesto ninguna CIVITAI_API_KEY. Las descargas podrían fallar.')

for url, dest, label in DOWNLOADS:
    if os.path.exists(dest):
        print(f'  ✓ {label} ya descargado')
        continue
    print(f'  ⬇ {label}...')
    try:
        with requests.get(url, headers=headers, stream=True, allow_redirects=True) as r:
            r.raise_for_status()
            with open(dest, 'wb') as f:
                for chunk in r.iter_content(chunk_size=8192):
                    f.write(chunk)
        print(f'  ✓ {label} listo')
    except Exception as e:
        print(f'  ❌ Error al descargar {label}: {e}')
        raise e

import torch
from diffusers import StableDiffusionXLPipeline
print('\n🔧 Cargando modelo en GPU (1-2 min)...')
pipe = StableDiffusionXLPipeline.from_single_file(
    MODEL_PATH, torch_dtype=torch.float16, use_safetensors=True,
).to('cuda')
pipe.enable_attention_slicing()
try:
    pipe.enable_xformers_memory_efficient_attention()
except Exception:
    pass

print('\n✅ Todo listo — ejecutá la Celda 2 para conectarte a la web.')

---

## ▶ Celda 2 — Conectar a la web y empezar a generar

Ejecutá esta celda. Automáticamente:
- Crea un túnel seguro de Cloudflare
- Se registra en **art.kxd.es** como GPU disponible
- Empieza a procesar pedidos de imagen

Cuando veas ✅ **SINCRONIZADO**, la GPU ya está online en la web.  
Si algo falla, también muestra el enlace del túnel para pegarlo manualmente en `/admin`.

> ⚠️ **No cerrés esta pestaña** mientras el Colab esté procesando imágenes.

In [ ]:
WEB_URL = 'https://art.kxd.es'

import io, base64, threading, time, re, subprocess
import requests
from fastapi import FastAPI
from typing import Optional, Dict
import uvicorn, torch
from PIL import Image

API_PORT = 7860
app_api = FastAPI()
generation_lock = threading.Lock()

def run_inference(job_dict):
    with generation_lock:
        # Limpiar siempre el estado de LoRAs anterior antes de cargar nuevos.
        # Sin esto, cargas sucesivas con adapter_name acumulan estado corrupto
        # y provocan 'list index out of range' en diffusers.
        try:
            pipe.unload_lora_weights()
        except Exception:
            pass

        lora_weights = job_dict.get('loraWeights') or {}
        active_loras, active_weights = [], []
        for name, weight in lora_weights.items():
            if name in LORA_REGISTRY and weight and weight > 0:
                adapter = f'lora_{name}'
                try:
                    pipe.load_lora_weights(
                        LORA_REGISTRY[name],
                        weight_name=os.path.basename(LORA_REGISTRY[name]),
                        adapter_name=adapter
                    )
                    active_loras.append(adapter)
                    active_weights.append(float(weight))
                except Exception as e:
                    print(f'  ⚠ LoRA {name}: {e}')
        if active_loras:
            try:
                pipe.set_adapters(active_loras, adapter_weights=active_weights)
            except Exception as e:
                print(f'  ⚠ Error aplicando adaptadores: {e}')

        seed = job_dict.get('seed')
        if seed is None:
            seed = int(torch.randint(0, 2**32, (1,)).item())
        gen = torch.Generator('cuda').manual_seed(seed)

        with torch.autocast('cuda'):
            result = pipe(
                prompt=job_dict.get('prompt', ''),
                negative_prompt=job_dict.get('negativePrompt') or 'nsfw, blurry, low quality, deformed',
                width=job_dict.get('width', 1024),
                height=job_dict.get('height', 1024),
                num_inference_steps=job_dict.get('steps', 30),
                guidance_scale=job_dict.get('cfgScale', 7.0),
                generator=gen,
            )
        if active_loras:
            try:
                pipe.unload_lora_weights()
            except Exception:
                pass

        return result.images[0], seed

threading.Thread(
    target=lambda: uvicorn.run(app_api, host='0.0.0.0', port=API_PORT, log_level='warning'),
    daemon=True
).start()
time.sleep(2)

print('🌐 Creando túnel Cloudflare...')
cf_proc = subprocess.Popen(
    ['cloudflared', 'tunnel', '--url', f'http://localhost:{API_PORT}', '--no-autoupdate'],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, universal_newlines=True
)

TUNNEL_URL = None
for _ in range(120):
    line = cf_proc.stdout.readline()
    match = re.search(r'https://[\w-]+\.trycloudflare\.com', line)
    if match:
        TUNNEL_URL = match.group(0)
        break

if not TUNNEL_URL:
    raise RuntimeError('No se pudo obtener la URL del túnel.')

SESSION_ID = None
try:
    resp = requests.post(f'{WEB_URL}/api/sessions/register',
        json={'tunnelUrl': TUNNEL_URL}, timeout=15)
    resp.raise_for_status()
    SESSION_ID = resp.json()['id']
    sync_ok = True
except Exception as e:
    sync_ok = False
    sync_error = str(e)

print()
print('━' * 52)
if sync_ok:
    print('✅  SINCRONIZADO — GPU activa en art.kxd.es')
    print(f'    Sesión #{SESSION_ID}')
else:
    print('⚠️  No se pudo conectar automáticamente')
    print(f'    Error: {sync_error}')
    print('👉  Pegá este enlace en /admin de la web:')
print(f'\n    {TUNNEL_URL}\n')
print('🐾  Procesando trabajos... Presioná ■ para detener.')
print('━' * 52)

def _afk():
    from IPython.display import display, Javascript
    while True:
        time.sleep(55)
        try:
            display(Javascript('document.dispatchEvent(new MouseEvent("mousemove",{bubbles:true}));'))
        except Exception:
            pass
threading.Thread(target=_afk, daemon=True).start()

HEADERS = {'Content-Type': 'application/json'}
start = time.time()
try:
    while time.time() - start < 12 * 3600:
        try:
            r = requests.get(f'{WEB_URL}/api/internal/next-job',
                params={'sessionId': SESSION_ID}, headers=HEADERS, timeout=10)
            r.raise_for_status()
            job = r.json().get('job')
        except Exception as e:
            print(f'⚠ Poll: {e}')
            time.sleep(3)
            continue

        if not job:
            time.sleep(3)
            continue

        jid = job['id']
        print(f'\n🎨 {jid[:8]}… | {job.get("width",1024)}×{job.get("height",1024)} | {job.get("steps",30)} pasos')
        print(f'   {str(job.get("prompt",""))[:80]}')

        try:
            t0 = time.time()
            img_obj, seed = run_inference(job)
            print(f'   ✓ {time.time()-t0:.1f}s  seed={seed}')

            # Guardar con compresión JPEG estándar al 80%
            buf = io.BytesIO()
            img_obj.save(buf, format='JPEG', quality=80)
            img_b64 = base64.b64encode(buf.getvalue()).decode()

            try:
                requests.post(f'{WEB_URL}/api/internal/job-complete',
                    headers=HEADERS, timeout=60,
                    json={'jobId': jid, 'sessionId': SESSION_ID, 'imageBase64': img_b64, 'seed': seed}
                ).raise_for_status()
            except requests.exceptions.HTTPError as http_err:
                # Ajuste súper agresivo a 512x512 y 50% de calidad si vuelve a dar error de tamaño máximo
                if http_err.response is not None and http_err.response.status_code == 413:
                    print('   ⚠ Servidor rechazó tamaño extremo. Forzando miniatura comprimida a 512x512...')
                    img_resized = img_obj.resize((512, 512), Image.Resampling.LANCZOS)
                    buf_small = io.BytesIO()
                    img_resized.save(buf_small, format='JPEG', quality=50)
                    img_b64_small = base64.b64encode(buf_small.getvalue()).decode()
                    
                    requests.post(f'{WEB_URL}/api/internal/job-complete',
                        headers=HEADERS, timeout=60,
                        json={'jobId': jid, 'sessionId': SESSION_ID, 'imageBase64': img_b64_small, 'seed': seed}
                    ).raise_for_status()
                else:
                    raise http_err
        except Exception as e:
            print(f'   ❌ {e}')
            try:
                requests.post(f'{WEB_URL}/api/internal/job-failed',
                    headers=HEADERS, timeout=10,
                    json={'jobId': jid, 'sessionId': SESSION_ID, 'errorMessage': str(e)})
            except Exception:
                pass

except KeyboardInterrupt:
    print('\n🛑 Detenido.')
finally:
    try:
        requests.post(f'{WEB_URL}/api/internal/session-end',
            headers=HEADERS, timeout=5, json={'sessionId': SESSION_ID})
    except Exception:
        pass
    if cf_proc.poll() is None:
        cf_proc.terminate()